# Reusable GRPO RL Single-Epoch Notebook - Qwen3.5-0.8B Cloud Classification Ranking

This notebook is configured to run **one RL epoch at a time**. Change only
`CURRENT_RL_EPOCH` in Section 2 before each run. All epoch-specific labels,
input adapter detection, output folders, file names, metrics, plots, and zip names
are derived from that variable.

Example: if `CURRENT_RL_EPOCH = 4`, the notebook loads the epoch-3 adapter,
runs RL epoch 4, evaluates epoch 4 on the test set, and saves the epoch-4 adapter.

**Reward:** a single weighted reward aligned with top-1 accuracy: `90%` top-1 reward + `10%` format reward.


In [ ]:
"""
Checks:
- Turn on internet
- Accelerator GPU T4 X2
- Persistence Files only
- Return from checkpoint
"""

## 1. Install Dependencies (run once, then restart kernel)

In [ ]:
pip show torch torchaudio torchvision

In [ ]:
"""
%pip uninstall -y torchaudio
%pip install --no-cache-dir --upgrade transformers peft trl
import transformers
import peft
import trl

print(transformers.__version__)
print(peft.__version__)
print(trl.__version__)
"""

In [ ]:
# Run this cell on a FRESH Kaggle image, then restart the kernel.
"""
import os
import subprocess
import sys

INSTALL_SENTINEL = "/kaggle/working/trl_install_restart_required.txt"
INSTALL_REPORT = "/kaggle/working/trl_install_report.txt"


def pip_install(*packages):
    cmd = [sys.executable, "-m", "pip", "install", "-U", "--no-cache-dir", *packages]
    print("Running:", " ".join(cmd))
    subprocess.check_call(cmd)


def pip_show_version(pkg: str):
    try:
        out = subprocess.check_output(
            [sys.executable, "-m", "pip", "show", pkg],
            text=True, stderr=subprocess.DEVNULL,
        )
    except subprocess.CalledProcessError:
        return None
    for line in out.splitlines():
        if line.lower().startswith("version:"):
            return line.split(":", 1)[1].strip()
    return None


subprocess.call([
    sys.executable, "-m", "pip", "uninstall", "-y",
    "verl", "vllm", "mistral-common", "mistral_common", "trl",
    "peft", "transformers", "tokenizers", "accelerate", "bitsandbytes",
])

pip_install("pip<25.1")
pip_install("git+https://github.com/huggingface/transformers.git")
pip_install(
    "trl>=0.17.0",
    "peft>=0.14.0",
    "accelerate>=1.0.0",
    "bitsandbytes>=0.45.0",
    "datasets>=3.0.0",
    "evaluate>=0.4.0",
    "safetensors>=0.4.0",
    "scikit-learn>=1.6.0",
    "seaborn>=0.13.0",
    "matplotlib>=3.8.0,<3.11.0",
    "pillow>=10.4.0,<12.0.0",
    "torchvision",
    "qwen-vl-utils>=0.0.10",
)
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--force-reinstall", "--no-cache-dir",
    "numpy==2.1.3",
    "scipy==1.14.1",
    "scikit-learn==1.7.2",
    "pillow>=10.4.0,<12.0.0",
])
subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"])

_pkg_versions = {
    pkg: pip_show_version(pkg)
    for pkg in ("transformers", "peft", "trl", "accelerate", "torch", "numpy", "pillow")
}
print("\n=== Section 1 final installed versions ===")
for _pkg, _ver in _pkg_versions.items():
    print(f"  {_pkg}: {_ver}")
print("=" * 44)

with open(INSTALL_REPORT, "w", encoding="utf-8") as handle:
    for _pkg, _ver in _pkg_versions.items():
        handle.write(f"{_pkg}={_ver}\n")

_missing = [pkg for pkg in ("transformers", "peft", "trl") if pip_show_version(pkg) is None]
if _missing:
    raise RuntimeError(f"Installation finished, but pip cannot find: {_missing}. Check the pip output above.")

with open(INSTALL_SENTINEL, "w", encoding="utf-8") as handle:
    handle.write(str(os.getpid()))

print(
    "\nDone. TRL GRPO dependencies are installed.\n"
    "IMPORTANT: use 'Run > Restart kernel' before continuing from section 2."
)
"""

## 2. CUDA Helpers, Device & Paths

In [ ]:
import ctypes
import glob
import importlib.util
import os
import re
import sys
import types

INSTALL_SENTINEL = "/kaggle/working/trl_install_restart_required.txt"
INSTALL_REPORT = "/kaggle/working/trl_install_report.txt"
_missing = [name for name in ("transformers", "peft", "trl") if importlib.util.find_spec(name) is None]
if _missing:
    if os.path.exists(INSTALL_REPORT):
        with open(INSTALL_REPORT, "r", encoding="utf-8") as _handle:
            _report_lines = _handle.read().strip()
        raise ModuleNotFoundError(
            f"Missing packages in this Kaggle session: {_missing}.\n"
            f"Section 1 previously wrote {INSTALL_REPORT}:\n{_report_lines}\n"
            "Re-run section 1, then restart only the kernel before running section 2."
        )
    raise ModuleNotFoundError(
        f"Missing packages in this Kaggle session: {_missing}. Run section 1 first, "
        "then restart only the kernel before running section 2."
    )
if os.path.exists(INSTALL_SENTINEL):
    with open(INSTALL_SENTINEL, "r", encoding="utf-8") as handle:
        install_pid = handle.read().strip()
    if install_pid == str(os.getpid()):
        raise RuntimeError("Dependencies were installed in this kernel, but the kernel has not been restarted yet. Use 'Run > Restart kernel' now.")
    os.remove(INSTALL_SENTINEL)

os.environ["PYTORCH_JIT"] = "0"
os.environ["TORCHDYNAMO_DISABLE"] = "1"
os.environ.setdefault("WANDB_DISABLED", "true")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

# Search all CUDA locations
import site as _site
_search_patterns = [
    "/usr/local/cuda*/lib64/libnvrtc-builtins.so*",
    "/usr/local/cuda*/lib64/libnvJitLink.so*",
    "/usr/lib/x86_64-linux-gnu/libnvrtc-builtins.so*",
    "/usr/lib/x86_64-linux-gnu/libnvJitLink.so*",
]
for _sp in _site.getsitepackages():
    _search_patterns.append(os.path.join(_sp, "nvidia", "*", "lib", "libnvrtc-builtins.so*"))
    _search_patterns.append(os.path.join(_sp, "nvidia", "*", "lib", "libnvJitLink.so*"))
for _pattern in _search_patterns:
    for _lib in sorted(glob.glob(_pattern)):
        try:
            ctypes.CDLL(_lib, mode=ctypes.RTLD_GLOBAL)
        except OSError:
            pass

import torch
import numpy as np

if hasattr(torch._C, "_jit_set_nvfuser_enabled"):
    torch._C._jit_set_nvfuser_enabled(False)
try:
    import triton.backends as _tb
    if not hasattr(_tb, "compiler"):
        _tb.compiler = types.SimpleNamespace(AttrsDescriptor=None)
except ImportError:
    pass

import transformers, peft, trl
print(f"torch: {torch.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"peft: {peft.__version__}")
print(f"trl: {trl.__version__}")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU: {torch.cuda.get_device_name(0)} ({gpu.total_memory / 1024**3:.1f} GB)")
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

# ── Paths ──────────────────────────────────────────────────────────
MODEL_NAME = "Qwen/Qwen3.5-0.8B"
DATASET_PATH = "/kaggle/input/datasets/nachomorerarespaldo2/clouddataset-fixed-10/CloudDataset_Fixed"
OUTPUT_DIR = "/kaggle/working/qwen35-08-cloud-classifier-grpo-lora-rl"
INITIAL_LORA_ADAPTER_PATH = "/kaggle/input/datasets/nachomorerarespaldo2/grpo-epoch1-adapter-fromepoch15"
GRPO_ADAPTER_DATASET_OWNER = "nachomorerarespaldo2"

# ── Single variable to change for each new RL epoch ────────────────
# ▶▶ SET THIS before each run. Everything else is derived from it.
CURRENT_RL_EPOCH = 1  # <-- change this number only
PREVIOUS_RL_EPOCH = CURRENT_RL_EPOCH - 1
CURRENT_EPOCH_LABEL = f"epoch-{CURRENT_RL_EPOCH}"
PREVIOUS_EPOCH_LABEL = f"epoch-{PREVIOUS_RL_EPOCH}"
CURRENT_RUN_NAME = f"trl_grpo_epoch{CURRENT_RL_EPOCH}"
PREVIOUS_RUN_NAME = f"trl_grpo_epoch{PREVIOUS_RL_EPOCH}"
CURRENT_ADAPTER_DIR_NAME = f"lora_adapter_epoch{CURRENT_RL_EPOCH}"
PREVIOUS_ADAPTER_DIR_NAME = f"lora_adapter_epoch{PREVIOUS_RL_EPOCH}"
GRPO_OUTPUT_DIR = os.path.join(OUTPUT_DIR, CURRENT_RUN_NAME)
ADAPTER_SAVE_DIR = os.path.join(GRPO_OUTPUT_DIR, CURRENT_ADAPTER_DIR_NAME)

# Epoch 1 starts from the supervised LoRA adapter. Later epochs start from the
# previous GRPO adapter uploaded as a Kaggle input dataset.
if CURRENT_RL_EPOCH == 1:
    ADAPTER_PATH = INITIAL_LORA_ADAPTER_PATH
else:
    ADAPTER_PATH = f"/kaggle/input/datasets/{GRPO_ADAPTER_DATASET_OWNER}/grpo-epoch{PREVIOUS_RL_EPOCH}-adapter"

from pathlib import Path
_inp = Path("/kaggle/input")
if _inp.exists():
    print("\n=== /kaggle/input datasets ===")
    for d in sorted(_inp.iterdir()):
        print(f"  {d.name}/")

    candidates = []
    if CURRENT_RL_EPOCH == 1:
        initial_adapter = Path(INITIAL_LORA_ADAPTER_PATH)
        if not (initial_adapter / "adapter_model.safetensors").exists():
            candidates = list(_inp.rglob("lora-epoch10-adapter/adapter_model.safetensors"))
            if not candidates:
                candidates = list(_inp.rglob("*lora*epoch10*/adapter_model.safetensors"))
    else:
        candidates = list(_inp.rglob(f"{PREVIOUS_RUN_NAME}/{PREVIOUS_ADAPTER_DIR_NAME}/adapter_model.safetensors"))
        if not candidates:
            candidates = list(_inp.rglob(f"{PREVIOUS_ADAPTER_DIR_NAME}/adapter_model.safetensors"))
        if not candidates:
            candidates = list(_inp.rglob(f"*epoch{PREVIOUS_RL_EPOCH}*/adapter_model.safetensors"))
    if candidates:
        ADAPTER_PATH = str(candidates[0].parent)
        print(f"\nAuto-detected input adapter: {ADAPTER_PATH}")

EVAL_SPLIT = "test"
MAX_LENGTH = 768
SEED = 42  # fixed seed so same 1500 train images every epoch
MAX_SAMPLES = None  # Set to e.g. 50 for a quick sanity check
EVAL_BATCH_SIZE = 4   # images per inference batch during evaluation
GRPO_MAX_TRAIN_SAMPLES = 1000  # reduced from 1500 → ~33% faster per epoch

torch.manual_seed(SEED)
np.random.seed(SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(GRPO_OUTPUT_DIR, exist_ok=True)
os.makedirs(ADAPTER_SAVE_DIR, exist_ok=True)

print(f"\nModel:   {MODEL_NAME}")
print(f"Dataset: {DATASET_PATH}")
print(f"Current RL epoch:  {CURRENT_RL_EPOCH}")
print(f"Previous RL epoch: {PREVIOUS_RL_EPOCH}")
print(f"Adapter input:     {ADAPTER_PATH}")
print(f"Training output:   {GRPO_OUTPUT_DIR}")
print(f"Adapter output:    {ADAPTER_SAVE_DIR}")

assert CURRENT_RL_EPOCH >= 1, "CURRENT_RL_EPOCH must be >= 1"
assert os.path.isdir(DATASET_PATH), f"Dataset not found: {DATASET_PATH}"
assert os.path.isdir(ADAPTER_PATH), (
    f"Adapter not found: {ADAPTER_PATH}\n"
    "For epoch 1, attach the supervised LoRA adapter dataset. For later epochs, "
    f"attach the {PREVIOUS_EPOCH_LABEL} GRPO adapter dataset as Kaggle input."
)


## 3. Load and Merge Dataset Classes

In [ ]:
from datasets import ClassLabel, load_dataset

print("Loading dataset...")
dataset = load_dataset(
    "imagefolder",
    data_files={
        "train": f"{DATASET_PATH}/train/**",
        "validation": f"{DATASET_PATH}/val/**",
        "test": f"{DATASET_PATH}/test/**",
    },
)

original_labels = dataset["train"].features["label"].names
print(f"Original classes ({len(original_labels)}): {original_labels}")

class_mapping = {
    "Patterned Clouds":   "Altocumulus",
    "Sky (General)":      "Clear Sky",
    "Thick Dark Clouds":  "Cumulonimbus",
    "Thick White Clouds": "Cumulus",
    "Veil Clouds":        "Veil",
}

merged_labels = sorted({class_mapping.get(label, label) for label in original_labels})
label2id = {label: i for i, label in enumerate(merged_labels)}
NUM_LABELS = len(merged_labels)

def remap_example(example):
    original_name = original_labels[example["label"]]
    merged_name = class_mapping.get(original_name, original_name)
    example["label"] = label2id[merged_name]
    return example

dataset = dataset.map(remap_example)
new_features = dataset["train"].features.copy()
new_features["label"] = ClassLabel(names=merged_labels)
dataset = dataset.cast(new_features)

new_labels = dataset["train"].features["label"].names
label2id = {label: i for i, label in enumerate(new_labels)}
id2label = {i: label for label, i in label2id.items()}

print(f"Merged classes ({NUM_LABELS}): {new_labels}")
print(f"Train: {len(dataset['train'])}, Val: {len(dataset['validation'])}, Test: {len(dataset['test'])}")


## 4. Ranking Prompts & Parsing

In [ ]:
import random

SYSTEM_PROMPT = (
    "You are a meteorological expert specialized in cloud classification. "
    "Given a cloud image, rank ALL cloud types from most probable to least probable. "
    f"Output a numbered list from 1 to {NUM_LABELS}, one class per line. "
    "Use exactly these class names: "
    + ", ".join(new_labels) + "."
)

USER_PROMPT = (
    "Rank the cloud types for this image from most likely to least likely. "
    f"Output a numbered list (1-{NUM_LABELS}) with one class per line."
)

VALID_CLASSES_LOWER = {label.lower(): label for label in new_labels}

def _parse_ranking(text):
    if isinstance(text, list):
        text = text[0]["content"] if text else ""
    lines = text.strip().split("\n")
    ranked = []
    for line in lines:
        line = line.strip()
        if not line:
            continue
        match = re.match(r'^\d+[.)\s]\s*(.+)$', line)
        if match:
            class_name = match.group(1).strip()
            if class_name.lower() in VALID_CLASSES_LOWER:
                ranked.append(VALID_CLASSES_LOWER[class_name.lower()])
            else:
                ranked.append(class_name)
    return ranked if ranked else None

def build_messages(label_text=None, seed=None):
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user", "content": [
            {"type": "image"},
            {"type": "text", "text": USER_PROMPT},
        ]},
    ]
    if label_text is not None:
        def build_ranking_string(correct_label, all_labels, seed=None):
            rng = random.Random(seed)
            others = [l for l in all_labels if l != correct_label]
            rng.shuffle(others)
            ranked = [correct_label] + others
            return "\n".join(f"{i+1}. {label}" for i, label in enumerate(ranked))
        ranking = build_ranking_string(label_text, new_labels, seed=seed)
        messages.append({"role": "assistant", "content": [{"type": "text", "text": ranking}]})
    return messages

def apply_template(messages, add_generation_prompt):
    return processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=add_generation_prompt,
    )

def get_top1_prediction(raw_text):
    ranked = _parse_ranking(raw_text)
    if ranked and ranked[0] in set(new_labels):
        return ranked[0]
    return "UNPARSEABLE"

def get_full_ranking(raw_text):
    ranked = _parse_ranking(raw_text)
    return ranked if ranked else []

print("Prompts & parsing ready.")

## 5. Load Base Model + Previous-Epoch LoRA Adapter

Load the base Qwen3.5-0.8B model and the LoRA adapter from `PREVIOUS_RL_EPOCH`.
Then **unfreeze** the adapter so `CURRENT_RL_EPOCH` can continue training.


In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor
from peft import PeftModel

print("Loading processor...")
processor = AutoProcessor.from_pretrained(MODEL_NAME)

compute_dtype = torch.bfloat16 if (DEVICE == "cuda" and torch.cuda.is_bf16_supported()) else torch.float32

print("Loading base model...")
base_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME,
    torch_dtype=compute_dtype,
    device_map="auto" if DEVICE == "cuda" else None,
    attn_implementation="flash_attention_2" if DEVICE == "cuda" else None,
)
if DEVICE != "cuda":
    base_model = base_model.to(DEVICE)

print(f"Loading {PREVIOUS_EPOCH_LABEL} adapter from {ADAPTER_PATH} ...")
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH, is_trainable=True)
model.print_trainable_parameters()
print(f"Adapter loaded and set to trainable for RL {CURRENT_EPOCH_LABEL}.")


## 6. Reward Function

Use one weighted reward so GRPO optimizes a single objective aligned with the
main evaluation metric:

`R = alpha * R_top1 + (1 - alpha) * R_format`, with `alpha = 0.9`.

`R_top1` is the main signal and gives `+1` only when the first ranked class is
the ground truth; all other parseable or unparseable answers receive `-1`.
`R_format` stays in `[0, 1]` and acts only as a soft constraint for valid
numbered lists with the expected classes and no duplicates.

In [ ]:
TOP1_REWARD_WEIGHT = 0.90
FORMAT_REWARD_WEIGHT = 1.0 - TOP1_REWARD_WEIGHT


def format_reward_single(completion):
    ranked = _parse_ranking(completion)
    if not ranked:
        return 0.0
    valid_names = set(new_labels)
    reward = 0.25  # parseable numbered list
    if len(ranked) == NUM_LABELS:
        reward += 0.25
    if all(c in valid_names for c in ranked):
        reward += 0.25
    if len(ranked) == len(set(ranked)):
        reward += 0.25
    return reward


def top1_reward_single(completion, ground_truth):
    ranked = _parse_ranking(completion)
    if not ranked:
        return -1.0
    return 1.0 if ranked[0] == ground_truth else -1.0


def ranking_reward_single(completion, ground_truth):
    return (
        TOP1_REWARD_WEIGHT * top1_reward_single(completion, ground_truth)
        + FORMAT_REWARD_WEIGHT * format_reward_single(completion)
    )


def ranking_reward(completions, ground_truth, **kwargs):
    return [
        ranking_reward_single(completion, gt)
        for completion, gt in zip(completions, ground_truth)
    ]


print(
    f"Reward function defined: {TOP1_REWARD_WEIGHT:.0%} top-1 "
    f"+ {FORMAT_REWARD_WEIGHT:.0%} format."
)

## 7. Build GRPO Dataset

In [ ]:
from datasets import Dataset as HFDataset

def build_grpo_dataset(hf_dataset):
    records = []
    for item in hf_dataset:
        try:
            img = item["image"].convert("RGB")
        except Exception:
            continue
        label_name = id2label[int(item["label"])]
        prompt_messages = [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
            {"role": "user", "content": [
                {"type": "image"},
                {"type": "text", "text": USER_PROMPT},
            ]},
        ]
        records.append({
            "prompt": prompt_messages,
            "image": img,
            "ground_truth": label_name,
        })
    return HFDataset.from_list(records)

print("Building GRPO dataset...")
grpo_train_ds = build_grpo_dataset(dataset["train"])
grpo_train_ds = grpo_train_ds.shuffle(seed=SEED)
if GRPO_MAX_TRAIN_SAMPLES is not None:
    grpo_train_ds = grpo_train_ds.select(range(min(GRPO_MAX_TRAIN_SAMPLES, len(grpo_train_ds))))
print(f"GRPO train samples: {len(grpo_train_ds)}")

## 8. Phase 2b - GRPO Current Epoch

This section runs exactly **one** RL epoch: `CURRENT_RL_EPOCH`.

To run the next epoch in a future notebook/session, change `CURRENT_RL_EPOCH` in
Section 2. The previous adapter input, output paths, file names, titles, and saved
artifacts are derived from that value.

The adapter is saved in `finally`, so if the run is interrupted after training progress,
the latest available adapter snapshot is still written to `ADAPTER_SAVE_DIR`.


In [ ]:
import glob
import os

# Safe rerun setup: Section 2 normally defines these values.
if "CURRENT_RL_EPOCH" not in globals():
    raise NameError("CURRENT_RL_EPOCH is not defined. Run Section 2 first and set CURRENT_RL_EPOCH there.")
if "OUTPUT_DIR" not in globals():
    raise NameError("OUTPUT_DIR is not defined. Run Section 2 before this cell.")
PREVIOUS_RL_EPOCH = CURRENT_RL_EPOCH - 1
CURRENT_EPOCH_LABEL = f"epoch-{CURRENT_RL_EPOCH}"
PREVIOUS_EPOCH_LABEL = f"epoch-{PREVIOUS_RL_EPOCH}"
CURRENT_RUN_NAME = f"trl_grpo_epoch{CURRENT_RL_EPOCH}"
CURRENT_ADAPTER_DIR_NAME = f"lora_adapter_epoch{CURRENT_RL_EPOCH}"
GRPO_OUTPUT_DIR = os.path.join(OUTPUT_DIR, CURRENT_RUN_NAME)
ADAPTER_SAVE_DIR = os.path.join(GRPO_OUTPUT_DIR, CURRENT_ADAPTER_DIR_NAME)
os.makedirs(GRPO_OUTPUT_DIR, exist_ok=True)
os.makedirs(ADAPTER_SAVE_DIR, exist_ok=True)

from trl import GRPOTrainer, GRPOConfig

training_cuda = DEVICE == "cuda"
training_bf16 = training_cuda and torch.cuda.is_bf16_supported()
training_fp16 = training_cuda and not training_bf16

grpo_config = GRPOConfig(
    output_dir=GRPO_OUTPUT_DIR,
    num_generations=4,
    generation_batch_size=4,  # must be divisible by num_generations
    max_completion_length=110,  # 11-class rankings fit in ~100 tokens; was 150
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=5e-6/CURRENT_RL_EPOCH,
    lr_scheduler_type="cosine",
    warmup_steps=5,
    bf16=training_bf16,
    fp16=training_fp16,
    logging_steps=5,
    save_strategy="steps",
    save_steps=25,
    save_total_limit=2,
    report_to="none",
    seed=SEED,
    log_completions=True,
    temperature=0.7,  # Force diverse completions for GRPO comparison.
    chat_template_kwargs={"enable_thinking": False},
)

print(f"GRPO RL {CURRENT_EPOCH_LABEL} - with temperature=0.7 and thinking disabled")
print(f"Training from {PREVIOUS_EPOCH_LABEL} adapter: {ADAPTER_PATH}")
print(f"Adapter will be saved to: {ADAPTER_SAVE_DIR}")
print(f"Reward function: {TOP1_REWARD_WEIGHT:.0%} top-1 + {FORMAT_REWARD_WEIGHT:.0%} format")

# Optional: remove old checkpoints first
"""
import shutil

if os.path.exists(GRPO_OUTPUT_DIR):
    shutil.rmtree(GRPO_OUTPUT_DIR)
"""

COMPLETIONS_DIR = os.path.join(GRPO_OUTPUT_DIR, "completions")
os.makedirs(COMPLETIONS_DIR, exist_ok=True)

# Resume from checkpoint if available for this current epoch.
_ckpts = sorted(
    glob.glob(os.path.join(GRPO_OUTPUT_DIR, "checkpoint-*")),
    key=lambda p: int(p.rsplit("-", 1)[-1]),
)
_resume_from = _ckpts[-1] if _ckpts else False

#_resume_from = False

if _resume_from:
    print(f"Resuming {CURRENT_EPOCH_LABEL} from checkpoint: {_resume_from}")
else:
    print(f"No prior checkpoint - training from {PREVIOUS_EPOCH_LABEL} adapter.")

In [ ]:
import glob
import os

# Safe rerun setup: Section 2 normally defines these values.
if "CURRENT_RL_EPOCH" not in globals():
    raise NameError("CURRENT_RL_EPOCH is not defined. Run Section 2 first and set CURRENT_RL_EPOCH there.")
if "OUTPUT_DIR" not in globals():
    raise NameError("OUTPUT_DIR is not defined. Run Section 2 before this cell.")
PREVIOUS_RL_EPOCH = CURRENT_RL_EPOCH - 1
CURRENT_EPOCH_LABEL = f"epoch-{CURRENT_RL_EPOCH}"
PREVIOUS_EPOCH_LABEL = f"epoch-{PREVIOUS_RL_EPOCH}"
CURRENT_RUN_NAME = f"trl_grpo_epoch{CURRENT_RL_EPOCH}"
CURRENT_ADAPTER_DIR_NAME = f"lora_adapter_epoch{CURRENT_RL_EPOCH}"
GRPO_OUTPUT_DIR = os.path.join(OUTPUT_DIR, CURRENT_RUN_NAME)
ADAPTER_SAVE_DIR = os.path.join(GRPO_OUTPUT_DIR, CURRENT_ADAPTER_DIR_NAME)
os.makedirs(GRPO_OUTPUT_DIR, exist_ok=True)
os.makedirs(ADAPTER_SAVE_DIR, exist_ok=True)

from trl import GRPOTrainer, GRPOConfig

training_cuda = DEVICE == "cuda"
training_bf16 = training_cuda and torch.cuda.is_bf16_supported()
training_fp16 = training_cuda and not training_bf16

grpo_config = GRPOConfig(
    output_dir=GRPO_OUTPUT_DIR,
    num_generations=4,
    generation_batch_size=4,  # must be divisible by num_generations
    max_completion_length=110,  # 11-class rankings fit in ~100 tokens; was 150
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=5e-6/CURRENT_RL_EPOCH,
    lr_scheduler_type="cosine",
    warmup_steps=5,
    bf16=training_bf16,
    fp16=training_fp16,
    logging_steps=5,
    save_strategy="steps",
    save_steps=25,
    save_total_limit=2,
    report_to="none",
    seed=SEED,
    log_completions=True,
    temperature=0.7,  # Force diverse completions for GRPO comparison.
    chat_template_kwargs={"enable_thinking": False},
)

print(f"GRPO RL {CURRENT_EPOCH_LABEL} - with temperature=0.7 and thinking disabled")
print(f"Training from {PREVIOUS_EPOCH_LABEL} adapter: {ADAPTER_PATH}")
print(f"Adapter will be saved to: {ADAPTER_SAVE_DIR}")
print(f"Reward function: {TOP1_REWARD_WEIGHT:.0%} top-1 + {FORMAT_REWARD_WEIGHT:.0%} format")

# Optional: remove old checkpoints first
"""
import shutil

if os.path.exists(GRPO_OUTPUT_DIR):
    shutil.rmtree(GRPO_OUTPUT_DIR)
"""

COMPLETIONS_DIR = os.path.join(GRPO_OUTPUT_DIR, "completions")
os.makedirs(COMPLETIONS_DIR, exist_ok=True)

# Resume from checkpoint if available for this current epoch.
_ckpts = sorted(
    glob.glob(os.path.join(GRPO_OUTPUT_DIR, "checkpoint-*")),
    key=lambda p: int(p.rsplit("-", 1)[-1]),
)
_resume_from = _ckpts[-1] if _ckpts else False

#_resume_from = False

if _resume_from:
    print(f"Resuming {CURRENT_EPOCH_LABEL} from checkpoint: {_resume_from}")
else:
    print(f"No prior checkpoint - training from {PREVIOUS_EPOCH_LABEL} adapter.")

grpo_trainer = GRPOTrainer(
    model=model,
    reward_funcs=[ranking_reward],
    args=grpo_config,
    train_dataset=grpo_train_ds,
    processing_class=processor,
)

_status = "not_started"
try:
    grpo_result = grpo_trainer.train(resume_from_checkpoint=_resume_from)
    grpo_trainer.save_state()
    _status = "finished"
    print(f"GRPO RL {CURRENT_EPOCH_LABEL} finished cleanly.")
except KeyboardInterrupt:
    _status = "interrupted"
    print("Interrupted - saving latest adapter snapshot before raising...")
    raise
except Exception as _err:
    _status = f"failed:{type(_err).__name__}"
    print(f"Error: {_err!r} - saving latest adapter snapshot before raising...")
    raise
finally:
    os.makedirs(ADAPTER_SAVE_DIR, exist_ok=True)
    model.save_pretrained(ADAPTER_SAVE_DIR)
    processor.save_pretrained(ADAPTER_SAVE_DIR)
    print(f"[{_status}] RL {CURRENT_EPOCH_LABEL} adapter saved to {ADAPTER_SAVE_DIR}")


## 9. Evaluate Current Saved RL Epoch on Test Set


In [ ]:
import glob
import os

# Safe rerun setup: Section 2 normally defines these values.
if "CURRENT_RL_EPOCH" not in globals():
    raise NameError("CURRENT_RL_EPOCH is not defined. Run Section 2 first and set CURRENT_RL_EPOCH there.")
if "OUTPUT_DIR" not in globals():
    raise NameError("OUTPUT_DIR is not defined. Run Section 2 before this cell.")
PREVIOUS_RL_EPOCH = CURRENT_RL_EPOCH - 1
CURRENT_EPOCH_LABEL = f"epoch-{CURRENT_RL_EPOCH}"
PREVIOUS_EPOCH_LABEL = f"epoch-{PREVIOUS_RL_EPOCH}"
CURRENT_RUN_NAME = f"trl_grpo_epoch{CURRENT_RL_EPOCH}"
CURRENT_ADAPTER_DIR_NAME = f"lora_adapter_epoch{CURRENT_RL_EPOCH}"
GRPO_OUTPUT_DIR = os.path.join(OUTPUT_DIR, CURRENT_RUN_NAME)
ADAPTER_SAVE_DIR = os.path.join(GRPO_OUTPUT_DIR, CURRENT_ADAPTER_DIR_NAME)
os.makedirs(GRPO_OUTPUT_DIR, exist_ok=True)
os.makedirs(ADAPTER_SAVE_DIR, exist_ok=True)

from tqdm.auto import tqdm
import gc


def move_to(batch, device):
    return {k: v.to(device) if hasattr(v, "to") else v for k, v in batch.items()}


GENERATION_KWARGS = {"max_new_tokens": 110, "do_sample": False}
_EVAL_BATCH_SIZE = globals().get("EVAL_BATCH_SIZE", 4)


def classify_batch(images):
    """Run inference on a batch of PIL images; returns list of raw text strings."""
    images = [img.convert("RGB") for img in images]
    prompt_text = apply_template(build_messages(), add_generation_prompt=True)
    inputs = processor(
        text=[prompt_text] * len(images),
        images=images,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )
    inputs = move_to(inputs, next(model.parameters()).device)
    with torch.inference_mode():
        gen_ids = model.generate(**inputs, **GENERATION_KWARGS)
    gen_ids = gen_ids[:, inputs["input_ids"].shape[1]:]
    return processor.batch_decode(gen_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)


def classify_image(image):
    return classify_batch([image])[0].strip()


# Make sure evaluation uses the saved current-epoch adapter.
if "grpo_trainer" in globals():
    del grpo_trainer
if "model" in globals():
    del model
if "base_model" in globals():
    del base_model
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

print(f"Loading saved RL {CURRENT_EPOCH_LABEL} adapter for evaluation: {ADAPTER_SAVE_DIR}")
eval_base_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME,
    torch_dtype=compute_dtype,
    device_map="auto" if DEVICE == "cuda" else None,
    attn_implementation="flash_attention_2" if DEVICE == "cuda" else None,
)
if DEVICE != "cuda":
    eval_base_model = eval_base_model.to(DEVICE)
model = PeftModel.from_pretrained(eval_base_model, ADAPTER_SAVE_DIR, is_trainable=False)
model.eval()

split = dataset[EVAL_SPLIT]
if MAX_SAMPLES is not None:
    split = split.select(range(min(MAX_SAMPLES, len(split))))

true_labels_list, pred_labels_list, raw_responses, rankings = [], [], [], []

items = list(split)
for batch_start in tqdm(range(0, len(items), _EVAL_BATCH_SIZE),
                        total=(len(items) + _EVAL_BATCH_SIZE - 1) // _EVAL_BATCH_SIZE,
                        desc=f"Evaluating {CURRENT_EPOCH_LABEL}"):
    batch = items[batch_start: batch_start + _EVAL_BATCH_SIZE]
    raw_list = classify_batch([ex["image"] for ex in batch])
    for ex, raw in zip(batch, raw_list):
        raw = raw.strip()
        top1 = get_top1_prediction(raw)
        ranking = get_full_ranking(raw)
        true_label = id2label[int(ex["label"])]
        true_labels_list.append(true_label)
        pred_labels_list.append(top1)
        raw_responses.append(raw)
        rankings.append(ranking)

accuracy_top1 = sum(t == p for t, p in zip(true_labels_list, pred_labels_list)) / len(true_labels_list)
top3_correct = sum(1 for t, r in zip(true_labels_list, rankings) if t in r[:3])
accuracy_top3 = top3_correct / len(true_labels_list)
top5_correct = sum(1 for t, r in zip(true_labels_list, rankings) if t in r[:5])
accuracy_top5 = top5_correct / len(true_labels_list)
valid_rankings = sum(1 for r in rankings if len(r) == NUM_LABELS)
format_rate = valid_rankings / len(rankings)

mrr_sum = 0.0
for true_label, ranking in zip(true_labels_list, rankings):
    if true_label in ranking:
        rank = ranking.index(true_label) + 1
        mrr_sum += 1.0 / rank
mrr = mrr_sum / len(true_labels_list)

print(f"\n{'='*50}")
print(f"GRPO RL {CURRENT_EPOCH_LABEL} - Evaluation ({EVAL_SPLIT})")
print(f"{'='*50}")
print(f"  Top-1 accuracy: {accuracy_top1:.4f}")
print(f"  Top-3 accuracy: {accuracy_top3:.4f}")
print(f"  Top-5 accuracy: {accuracy_top5:.4f}")
print(f"  MRR:            {mrr:.4f}")
print(f"  Format rate:    {format_rate:.4f}")
print(f"  Samples:        {len(true_labels_list)}")
print(f"{'='*50}")


## 10. Classification Report & Confusion Matrix

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix

report = classification_report(
    true_labels_list, pred_labels_list, labels=new_labels, digits=3, zero_division=0
)
print(f"\nClassification report - RL {CURRENT_EPOCH_LABEL}")
print(report)

bal_acc = balanced_accuracy_score(true_labels_list, pred_labels_list)
print(f"Balanced accuracy (top-1): {bal_acc:.4f}")

cm = confusion_matrix(true_labels_list, pred_labels_list, labels=new_labels)
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=new_labels, yticklabels=new_labels, linewidths=0.5, ax=ax)
ax.set_xlabel("Predicted (Top-1)")
ax.set_ylabel("True")
ax.set_title(f"Confusion matrix - RL {CURRENT_EPOCH_LABEL} | top-1={accuracy_top1:.3f} | MRR={mrr:.3f} | split={EVAL_SPLIT}")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
cm_path = os.path.join(OUTPUT_DIR, f"confusion_matrix_rl_epoch{CURRENT_RL_EPOCH}_{EVAL_SPLIT}.png")
plt.savefig(cm_path, dpi=150)
plt.show()

predictions_df = pd.DataFrame({
    "true_label": true_labels_list,
    "predicted_top1": pred_labels_list,
    "ranking_top3": [(" > ").join(r[:3]) if len(r) >= 3 else "" for r in rankings],
    "ranking_full": [(" > ").join(r) if r else "" for r in rankings],
    "raw_response": raw_responses,
})
pred_path = os.path.join(OUTPUT_DIR, f"predictions_rl_epoch{CURRENT_RL_EPOCH}_{EVAL_SPLIT}.csv")
predictions_df.to_csv(pred_path, index=False)
print(f"Saved predictions to {pred_path}")

positions = []
for true_label, ranking in zip(true_labels_list, rankings):
    if true_label in ranking:
        positions.append(ranking.index(true_label) + 1)
    else:
        positions.append(12)

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(positions, bins=range(1, 14), edgecolor="black", alpha=0.7, align="left")
ax.set_xlabel("Position of correct class in ranking")
ax.set_ylabel("Count")
ax.set_title(f"Distribution of correct-class position - RL {CURRENT_EPOCH_LABEL} ({EVAL_SPLIT})")
ax.set_xticks(range(1, 13))
ax.set_xticklabels([str(i) for i in range(1, 12)] + ["miss"])
plt.tight_layout()
pos_path = os.path.join(OUTPUT_DIR, f"position_distribution_epoch{CURRENT_RL_EPOCH}_{EVAL_SPLIT}.png")
plt.savefig(pos_path, dpi=150)
plt.show()

print(f"\nPosition stats - RL {CURRENT_EPOCH_LABEL}:")
print(f"  Rank 1 (correct): {positions.count(1)} / {len(positions)} ({positions.count(1)/len(positions):.3f})")
print(f"  Rank 1-3:         {sum(1 for p in positions if p <= 3)} / {len(positions)}")
print(f"  Rank 1-5:         {sum(1 for p in positions if p <= 5)} / {len(positions)}")
print(f"  Missing:          {positions.count(12)} / {len(positions)}")


## 11. Save Artifacts

In [ ]:
import shutil

metrics_path = os.path.join(OUTPUT_DIR, f"metrics_rl_epoch{CURRENT_RL_EPOCH}_{EVAL_SPLIT}.txt")
with open(metrics_path, "w", encoding="utf-8") as f:
    f.write(f"GRPO RL {CURRENT_EPOCH_LABEL} - Evaluation ({EVAL_SPLIT})\n")
    f.write(f"{'='*50}\n")
    f.write(f"Top-1 accuracy: {accuracy_top1:.4f}\n")
    f.write(f"Top-3 accuracy: {accuracy_top3:.4f}\n")
    f.write(f"Top-5 accuracy: {accuracy_top5:.4f}\n")
    f.write(f"MRR:            {mrr:.4f}\n")
    f.write(f"Balanced acc:   {bal_acc:.4f}\n")
    f.write(f"Format rate:    {format_rate:.4f}\n")
    f.write(f"Samples:        {len(true_labels_list)}\n\n")
    f.write(report)
print(f"Metrics saved to {metrics_path}")

adapter_zip = os.path.join(OUTPUT_DIR, f"grpo_epoch{CURRENT_RL_EPOCH}_adapter")
if os.path.isdir(ADAPTER_SAVE_DIR):
    shutil.make_archive(adapter_zip, "zip", ADAPTER_SAVE_DIR)
    print(f"Adapter zip: {adapter_zip}.zip")

print(f"\nDone! RL {CURRENT_EPOCH_LABEL} weights, metrics, predictions, and plots are saved.")


In [ ]:
from pathlib import Path

path = Path('/kaggle/working/qwen35-08-cloud-classifier-grpo-lora-rl')
for p in path.iterdir():
    print(p.name)